# 7.11 預訓練模型怎麼學會對話？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩種遮罩：看得見，不一定直接考它**

X 的 assistant 格預測 A；PAD 不當答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/masks.svg")))

**先想一想：**只改prompt而不更新權重，叫SFT嗎？

預訓練學習文本分佈，SFT繼續用正確問答讓角色邊界後出現合適答案。架構不必換，但必須保存匹配的基模和tokenizer。

**把直覺寫成數學：**同一CE目標、不同輸入分佈和監督mask；SFT不是查資料表直接輸出答案。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import toy_conversations, render_chat, pad_batch
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss

x, y, valid = pad_batch([render_chat(m) for m in toy_conversations()[:2]])
model = TinyLM(ModelConfig(width=8))
loss = masked_loss(model(x, valid=valid)["logits"], y)
loss.backward()
print("SFT接口loss", loss.item())
print("GPU入口：python scripts/train.py --task sft --checkpoint checkpoints/text.pt --train --steps 300")

**如何讀結果：**這格確認資料到梯度，沒有聲稱隨機模型會算加法。前後回答要由真實checkpoint生成。

**只改一件事：**只把其中一例答案錯標，追蹤它的監督ID。
